# Lotus Care Clinic Data Warehouse
## Coursework Part C – ETL Notebook

This notebook profiles, cleans and integrates the raw clinic data supplied for Coursework Part C.

The raw CSV files are read from the `data` folder. No raw CSV file is edited manually.

The notebook will:
1. Profile the raw data.
2. Identify and clean data-quality problems.
3. Remove duplicate bookings according to the required grain.
4. Standardise dates and specialty values.
5. Validate the measures.
6. Integrate the appointment data with the dimension data.
7. Generate surrogate keys.
8. Export cleaned dimension and fact CSV files for loading into PostgreSQL.

### Import required library

This cell imports pandas, which will be used to read the CSV files and profile, clean and transform the data.

Expected output: No output should appear if the import is successful.

In [44]:
import pandas as pd

### Load the raw CSV files

This cell reads the five supplied CSV files from the `data` folder without modifying them.

Expected output: Five pandas DataFrames will be created:
- `appointments`
- `doctors`
- `clinics`
- `patients`
- `patient_updates`

In [45]:
appointments = pd.read_csv("data/appointments_raw.csv")
doctors = pd.read_csv("data/doctors.csv")
clinics = pd.read_csv("data/clinics.csv")
patients = pd.read_csv("data/patients.csv")
patient_updates = pd.read_csv("data/patient_updates.csv")

### Verify the raw row counts

This cell checks that the number of rows loaded from each CSV matches the counts provided in the coursework README.

Expected output:
- appointments: 1,524 rows
- doctors: 21 rows
- clinics: 5 rows
- patients: 300 rows
- patient_updates: 3 rows

In [46]:
print("Appointments:", len(appointments))
print("Doctors:", len(doctors))
print("Clinics:", len(clinics))
print("Patients:", len(patients))
print("Patient updates:", len(patient_updates))

Appointments: 1524
Doctors: 21
Clinics: 5
Patients: 300
Patient updates: 3


### Profile the structure of the raw data

This cell uses pandas `info()` to inspect the columns, data types and non-null counts in each raw dataset.

Expected output: The structure of each DataFrame will be displayed, including column names, data types and the number of non-null values.

In [47]:
appointments.info()


<class 'pandas.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   appointment_id        1524 non-null   str    
 1   patient_id            1524 non-null   str    
 2   doctor_id             1524 non-null   str    
 3   clinic_id             1524 non-null   str    
 4   appointment_date      1524 non-null   str    
 5   wait_minutes          1524 non-null   int64  
 6   consultation_minutes  1524 non-null   int64  
 7   fee                   1509 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 135.6 KB


### Profile all dimension and update files

This cell uses `info()` on the remaining raw DataFrames to inspect their structures and identify missing values or unexpected data types.

Expected output: The structure and non-null counts of the doctors, clinics, patients and patient update datasets.

In [48]:
appointments.info()
doctors.info()
clinics.info()
patients.info()
patient_updates.info()

<class 'pandas.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   appointment_id        1524 non-null   str    
 1   patient_id            1524 non-null   str    
 2   doctor_id             1524 non-null   str    
 3   clinic_id             1524 non-null   str    
 4   appointment_date      1524 non-null   str    
 5   wait_minutes          1524 non-null   int64  
 6   consultation_minutes  1524 non-null   int64  
 7   fee                   1509 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 135.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   doctor_id    21 non-null     str  
 1   doctor_name  21 non-null     str  
 2   specialty    21 non-null     str  
dtypes: str(3)
memory usage: 1.3 KB
<class '

### Check for missing values

This cell checks every column in the raw datasets for missing values using pandas `isnull().sum()`.

Expected output: The number of missing values in each column will be displayed. This will help identify fields that require cleaning before the data is loaded into the warehouse.

In [49]:
print("Appointments - missing values")
print(appointments.isnull().sum())

print("\nDoctors - missing values")
print(doctors.isnull().sum())

print("\nClinics - missing values")
print(clinics.isnull().sum())

print("\nPatients - missing values")
print(patients.isnull().sum())

print("\nPatient updates - missing values")
print(patient_updates.isnull().sum())

Appointments - missing values
appointment_id           0
patient_id               0
doctor_id                0
clinic_id                0
appointment_date         0
wait_minutes             0
consultation_minutes     0
fee                     15
dtype: int64

Doctors - missing values
doctor_id      0
doctor_name    0
specialty      0
dtype: int64

Clinics - missing values
clinic_id      0
clinic_name    0
city           0
province       0
dtype: int64

Patients - missing values
patient_id       0
gender           0
date_of_birth    0
district         0
dtype: int64

Patient updates - missing values
patient_id        0
new_district      0
effective_date    0
dtype: int64


In [66]:
print("Negative wait_minutes:", (appointments["wait_minutes"] < 0).sum())
print("Negative consultation_minutes:", (appointments["consultation_minutes"] < 0).sum())

Negative wait_minutes: 8
Negative consultation_minutes: 0


### Check for completely duplicated rows

This cell checks whether any rows are completely identical across all columns using pandas `duplicated()`.

Expected output: The number of completely duplicated rows will be displayed. This is only an initial check because repeated bookings may differ in one or more non-grain fields.

In [50]:
print("Completely duplicated appointment rows:", appointments.duplicated().sum())
print("Completely duplicated doctor rows:", doctors.duplicated().sum())
print("Completely duplicated clinic rows:", clinics.duplicated().sum())
print("Completely duplicated patient rows:", patients.duplicated().sum())
print("Completely duplicated patient update rows:", patient_updates.duplicated().sum())

Completely duplicated appointment rows: 0
Completely duplicated doctor rows: 0
Completely duplicated clinic rows: 0
Completely duplicated patient rows: 0
Completely duplicated patient update rows: 0


### Check repeated bookings using the fact-table grain

The fact-table grain is one appointment for one patient, one doctor, one clinic and one date. This cell checks for repeated bookings using those grain columns rather than comparing every column.

Expected output: The number of rows that belong to repeated booking groups will be displayed.

In [51]:
grain_columns = [
    "patient_id",
    "doctor_id",
    "clinic_id",
    "appointment_date"
]

grain_duplicates = appointments.duplicated(
    subset=grain_columns,
    keep=False
)

print("Rows involved in repeated bookings:", grain_duplicates.sum())

Rows involved in repeated bookings: 48


### Inspect repeated booking groups

This cell displays all appointment rows that share the same patient, doctor, clinic and appointment date.

Expected output: The 48 rows identified as belonging to repeated booking groups will be displayed, allowing the differences between repeated records to be examined before cleaning.

In [52]:
appointments.loc[
    grain_duplicates,
    grain_columns + [
        "appointment_id",
        "wait_minutes",
        "consultation_minutes",
        "fee"
    ]
].sort_values(grain_columns)

,patient_id,doctor_id,clinic_id,appointment_date,appointment_id,wait_minutes,consultation_minutes,fee
308,P0022,D14,C04,2025-12-20,A01462,3,18,4500.0
1379,P0022,D14,C04,2025-12-20,A01523,3,18,4500.0
562,P0028,D19,C04,2025-06-07,A00651,36,30,2500.0
810,P0028,D19,C04,2025-06-07,A01503,36,30,2500.0
377,P0075,D10,C02,2025-08-18,A00966,13,25,3000.0
429,P0075,D10,C02,2025-08-18,A01512,13,25,3000.0
1186,P0075,D10,C02,2025-11-20,A01522,93,32,3000.0
1422,P0075,D10,C02,2025-11-20,A01343,93,32,3000.0
906,P0081,D01,C03,2025-06-28,A01507,55,17,2500.0
1395,P0081,D01,C03,2025-06-28,A00752,55,17,2500.0


### Count the repeated booking groups

This cell counts the number of unique combinations of the grain columns that occur more than once.

Expected output: The number of distinct repeated booking groups will be shown. This may be smaller than the 48 affected rows because a group can contain more than two rows.

In [53]:
duplicate_groups = (
    appointments[
        appointments.duplicated(subset=grain_columns, keep=False)
    ]
    .groupby(grain_columns)
    .size()
    .sort_values(ascending=False)
)

print("Number of repeated booking groups:", len(duplicate_groups))
print("\nRows per repeated booking group:")
print(duplicate_groups)

Number of repeated booking groups: 24

Rows per repeated booking group:
patient_id  doctor_id  clinic_id  appointment_date
P0022       D14        C04        2025-12-20          2
P0028       D19        C04        2025-06-07          2
P0075       D10        C02        2025-08-18          2
                                  2025-11-20          2
P0081       D01        C03        2025-06-28          2
P0085       D07        C02        2025-08-22          2
P0095       D05        C05        2025-06-27          2
P0110       D01        C03        2025-01-25          2
P0118       D05        C05        2025-11-14          2
P0147       D15        C05        2025-09-06          2
P0158       D18        C03        2025-12-30          2
P0179       D19        C01        2025-07-17          2
P0182       D19        C01        2025-09-05          2
P0189       D13        C03        2025-10-30          2
P0196       D14        C04        2025-09-22          2
P0200       D11        C01        202

### Remove repeated bookings using the fact-table grain

The required grain is one booking for one patient, one doctor, one clinic and one appointment date. The profiling identified 24 repeated booking groups containing 48 rows. Each group contains two records with identical measure values, so one record from each repeated group can be retained.

Expected output: The appointment row count should decrease from 1,524 to 1,500, with 24 rows removed.

In [54]:
appointments_clean = appointments.drop_duplicates(
    subset=grain_columns,
    keep="first"
).copy()

print("Rows before duplicate removal:", len(appointments))
print("Rows after duplicate removal:", len(appointments_clean))
print("Rows removed:", len(appointments) - len(appointments_clean))

Rows before duplicate removal: 1524
Rows after duplicate removal: 1500
Rows removed: 24


### Verify duplicate removal

This cell checks the cleaned appointment data again using the fact-table grain.

Expected output: Zero rows should belong to repeated booking groups after duplicate removal.

In [55]:
remaining_duplicates = appointments_clean.duplicated(
    subset=grain_columns,
    keep=False
)

print(
    "Rows involved in repeated bookings after cleaning:",
    remaining_duplicates.sum()
)

Rows involved in repeated bookings after cleaning: 0


### Profile doctor specialties

This cell uses `value_counts()` to inspect the distinct specialty values in the raw doctors data. The purpose is to identify differences in spelling, capitalisation or whitespace before standardisation.

Expected output: A frequency table of the specialty values currently present in `doctors.csv`.

In [56]:
doctors["specialty"].value_counts(dropna=False)

specialty
General Medicine     4
Paediatrics          3
Cardiology           2
Orthopaedics         2
Neurology            2
Dermatology          2
dermatology          1
CARDIOLOGY           1
 Orthopaedics        1
cardiology           1
Neurology            1
general medicine     1
Name: count, dtype: int64

### Standardize specialty values

The specialty field contains inconsistent capitalization and leading spaces. These values need to be standardized so that the same specialty is represented consistently.

In [57]:
doctors["specialty"] = (
    doctors["specialty"]
    .str.strip()
    .str.title()
)

print("Distinct specialty values after standardization:")
print(doctors["specialty"].value_counts())

Distinct specialty values after standardization:
specialty
General Medicine    5
Cardiology          4
Dermatology         3
Paediatrics         3
Orthopaedics        3
Neurology           3
Name: count, dtype: int64


### Check appointment measures

The appointment measures are checked for missing and potentially invalid values. Waiting time and consultation time should not be negative, and the appointment fee should not be negative.

In [58]:
print("Missing values:")
print(appointments_clean[
    ["wait_minutes", "consultation_minutes", "fee"]
].isna().sum())

print("\nNegative values:")
print(
    (appointments_clean[
        ["wait_minutes", "consultation_minutes", "fee"]
    ] < 0).sum()
)

Missing values:
wait_minutes             0
consultation_minutes     0
fee                     15
dtype: int64

Negative values:
wait_minutes            8
consultation_minutes    0
fee                     0
dtype: int64


### Inspect invalid measure values

The profiling step identified 8 negative waiting times and 15 missing fees. The affected appointment records are inspected before deciding how to handle these values.

In [59]:
invalid_measures = appointments_clean[
    (appointments_clean["wait_minutes"] < 0) |
    (appointments_clean["fee"].isna())
]

print("Rows with missing or invalid measures:", len(invalid_measures))

display(
    invalid_measures[
        [
            "appointment_id",
            "patient_id",
            "doctor_id",
            "clinic_id",
            "appointment_date",
            "wait_minutes",
            "consultation_minutes",
            "fee"
        ]
    ]
)

Rows with missing or invalid measures: 23


,appointment_id,patient_id,doctor_id,clinic_id,appointment_date,wait_minutes,consultation_minutes,fee
50,A01332,P0220,D14,C04,2025-11-18,43,14,NaN
51,A01421,P0278,D14,C04,2025-12-08,7,33,NaN
52,A00537,P0223,D11,C01,2025-05-14,1,17,NaN
89,A01148,P0031,D13,C03,2025-10-01,1,25,NaN
179,A00720,P0174,D02,C02,2025-06-23,61,14,NaN
387,A01192,P0223,D01,C03,2025-10-15,-9,31,2500.0
452,A00837,P0217,D12,C02,2025-07-18,89,25,NaN
511,A01446,P0005,D18,C03,2025-12-17,-7,31,5000.0
608,A01024,P0160,D03,C03,2025-08-30,4,16,NaN
659,A01464,P0183,D06,C01,2025-12-22,-23,25,5000.0


### Define rules for invalid measures

Missing fees are replaced using the median of the valid fee values. Negative waiting times are replaced using the median of the non-negative waiting-time values. Consultation times do not require correction because no missing or negative values were found.

In [60]:
valid_fee_median = appointments_clean["fee"].median()

valid_wait_median = appointments_clean.loc[
    appointments_clean["wait_minutes"] >= 0,
    "wait_minutes"
].median()

print("Median valid fee:", valid_fee_median)
print("Median valid waiting time:", valid_wait_median)

Median valid fee: 3500.0
Median valid waiting time: 24.0


### Apply measure cleaning rules

Missing fee values are replaced with the median valid fee of 3500.0. Negative waiting times are replaced with the median valid waiting time of 24.0 minutes. The number of affected rows is recorded before and after the changes.

In [61]:
# Record the number of invalid values before cleaning
missing_fee_before = appointments_clean["fee"].isna().sum()
negative_wait_before = (appointments_clean["wait_minutes"] < 0).sum()

# Apply cleaning rules
appointments_clean.loc[
    appointments_clean["fee"].isna(), "fee"
] = valid_fee_median

appointments_clean.loc[
    appointments_clean["wait_minutes"] < 0, "wait_minutes"
] = valid_wait_median

# Verify after cleaning
missing_fee_after = appointments_clean["fee"].isna().sum()
negative_wait_after = (appointments_clean["wait_minutes"] < 0).sum()

print("Measure cleaning summary")
print("------------------------")
print("Missing fees before cleaning:", missing_fee_before)
print("Missing fees after cleaning:", missing_fee_after)
print("Negative waiting times before cleaning:", negative_wait_before)
print("Negative waiting times after cleaning:", negative_wait_after)

Measure cleaning summary
------------------------
Missing fees before cleaning: 15
Missing fees after cleaning: 0
Negative waiting times before cleaning: 8
Negative waiting times after cleaning: 0


In [62]:
print("Cleaning summary:")
print("Missing fee values identified before cleaning: 15")
print("Negative wait values identified before cleaning: 8")
print("Missing fee values after cleaning:", appointments_clean["fee"].isna().sum())
print("Negative wait values after cleaning:",
      (appointments_clean["wait_minutes"] < 0).sum())

Cleaning summary:
Missing fee values identified before cleaning: 15
Negative wait values identified before cleaning: 8
Missing fee values after cleaning: 0
Negative wait values after cleaning: 0


### Integrate appointments with doctor information

The cleaned appointment data is joined with the doctor dimension using `doctor_id`. A left join is used so that every appointment is retained while checking whether a matching doctor exists.

In [63]:
appointments_before_join = len(appointments_clean)

appointments_doctors = appointments_clean.merge(
    doctors,
    on="doctor_id",
    how="left",
    validate="many_to_one"
)

appointments_after_join = len(appointments_doctors)

print("Appointments before doctor join:", appointments_before_join)
print("Rows after doctor join:", appointments_after_join)
print("Rows changed:", appointments_after_join - appointments_before_join)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 doctor_id
      D07 ...

In [64]:
duplicate_doctor_ids = doctors[
    doctors.duplicated(
        subset=["doctor_id"],
        keep=False
    )
].sort_values("doctor_id")

print("Rows involved in duplicate doctor IDs:",
      len(duplicate_doctor_ids))

display(duplicate_doctor_ids)

Rows involved in duplicate doctor IDs: 2


,doctor_id,doctor_name,specialty
6,D07,Dr. Tharindu Rajapaksha,General Medicine
20,D07,Dr. Tharindu Rajapaksha,General Medicine


In [65]:
print(
    "Number of exact duplicate doctor rows:",
    doctors.duplicated().sum()
)

Number of exact duplicate doctor rows: 1


### Fix duplicate doctor record

One exact duplicate doctor row was found for doctor D07. Since both records contain identical doctor information, the duplicate row is removed from the working doctor DataFrame.

In [24]:
doctors_clean = doctors.drop_duplicates().copy()

print("Doctors before removing duplicates:", len(doctors))
print("Doctors after removing duplicates:", len(doctors_clean))
print("Exact duplicate rows remaining:", doctors_clean.duplicated().sum())

Doctors before removing duplicates: 21
Doctors after removing duplicates: 20
Exact duplicate rows remaining: 0


In [25]:
appointments_before_join = len(appointments_clean)

appointments_doctors = appointments_clean.merge(
    doctors_clean,
    on="doctor_id",
    how="left",
    validate="many_to_one"
)

appointments_after_join = len(appointments_doctors)

print("Appointments before doctor join:", appointments_before_join)
print("Rows after doctor join:", appointments_after_join)
print("Rows changed:", appointments_after_join - appointments_before_join)

Appointments before doctor join: 1500
Rows after doctor join: 1500
Rows changed: 0


In [26]:
missing_doctor_matches = appointments_doctors["doctor_name"].isna().sum()

print("Appointments without a matching doctor:",
      missing_doctor_matches)

Appointments without a matching doctor: 0


In [27]:
print("Clean appointments:", len(appointments_clean))
print("Clean doctors:", len(doctors_clean))
print("Clinics:", len(clinics))
print("Patients:", len(patients))

Clean appointments: 1500
Clean doctors: 20
Clinics: 5
Patients: 300


### Create dim_patient with surrogate keys

A surrogate key is generated for each patient using a sequential integer. The original `patient_id` is retained as a source identifier for traceability.

In [28]:
dim_patient = patients.copy()

dim_patient.insert(
    0,
    "patient_key",
    range(1, len(dim_patient) + 1)
)

print("dim_patient rows:", len(dim_patient))
display(dim_patient.head())

dim_patient rows: 300


,patient_key,patient_id,gender,date_of_birth,district
0,1,P0001,F,1980-06-21,Kurunegala
1,2,P0002,F,2006-12-04,Gampaha
2,3,P0003,M,1959-02-14,Galle
3,4,P0004,M,1990-02-09,Matara
4,5,P0005,F,1975-01-20,Colombo


### Create dim_doctor with surrogate keys

A surrogate key is generated for each unique doctor. The original doctor ID is retained as the source identifier, while the standardized specialty values are used in the dimension.

In [29]:
dim_doctor = doctors_clean.copy()

dim_doctor.insert(
    0,
    "doctor_key",
    range(1, len(dim_doctor) + 1)
)

print("dim_doctor rows:", len(dim_doctor))
display(dim_doctor.head())

dim_doctor rows: 20


,doctor_key,doctor_id,doctor_name,specialty
0,1,D01,Dr. Anil Perera,General Medicine
1,2,D02,Dr. Dilani Silva,Cardiology
2,3,D03,Dr. Kasun Fernando,Dermatology
3,4,D04,Dr. Nimali Jayasinghe,Paediatrics
4,5,D05,Dr. Ruwan Wickramasinghe,Orthopaedics


### Create dim_clinic with surrogate keys

A surrogate key is generated for each clinic using a sequential integer. The original clinic ID is retained as the source identifier.

In [30]:
dim_clinic = clinics.copy()

dim_clinic.insert(
    0,
    "clinic_key",
    range(1, len(dim_clinic) + 1)
)

print("dim_clinic rows:", len(dim_clinic))
display(dim_clinic)

dim_clinic rows: 5


,clinic_key,clinic_id,clinic_name,city,province
0,1,C01,Lotus Care Colombo,Colombo,Western
1,2,C02,Lotus Care Kandy,Kandy,Central
2,3,C03,Lotus Care Galle,Galle,Southern
3,4,C04,Lotus Care Kurunegala,Kurunegala,North Western
4,5,C05,Lotus Care Jaffna,Jaffna,Northern


### Create dim_date

The date dimension contains one row for every calendar date in 2025. It provides calendar attributes that can be used for time-based analysis such as yearly, quarterly, and monthly reporting.

In [31]:
date_range = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="D"
)

dim_date = pd.DataFrame({
    "date_key": date_range.strftime("%Y%m%d").astype(int),
    "full_date": date_range,
    "year": date_range.year,
    "quarter": "Q" + date_range.quarter.astype(str),
    "month": date_range.month,
    "month_name": date_range.month_name(),
    "day": date_range.day,
    "day_name": date_range.day_name()
})

print("dim_date rows:", len(dim_date))
display(dim_date.head())

dim_date rows: 365


,date_key,full_date,year,quarter,month,month_name,day,day_name
0,20250101,2025-01-01,2025,Q1,1,January,1,Wednesday
1,20250102,2025-01-02,2025,Q1,1,January,2,Thursday
2,20250103,2025-01-03,2025,Q1,1,January,3,Friday
3,20250104,2025-01-04,2025,Q1,1,January,4,Saturday
4,20250105,2025-01-05,2025,Q1,1,January,5,Sunday


### Create fact_appointment

The fact table represents one appointment at the defined grain: one patient with one doctor at one clinic on one date. The table stores dimension surrogate keys and the three appointment measures. Descriptive attributes are kept in the dimension tables.

In [32]:
patient_key_map = dim_patient.set_index("patient_id")["patient_key"]
doctor_key_map = dim_doctor.set_index("doctor_id")["doctor_key"]
clinic_key_map = dim_clinic.set_index("clinic_id")["clinic_key"]
date_key_map = dim_date.set_index("full_date")["date_key"]

In [33]:
# Standardize appointment dates to datetime
appointments_clean["appointment_date"] = pd.to_datetime(
    appointments_clean["appointment_date"],
    format="mixed",
    dayfirst=False
)

print("Date conversion successful.")
print("Invalid/missing dates:", appointments_clean["appointment_date"].isna().sum())
print("Date range:")
print(appointments_clean["appointment_date"].min())
print(appointments_clean["appointment_date"].max())

Date conversion successful.
Invalid/missing dates: 0
Date range:
2025-01-01 00:00:00
2025-12-31 00:00:00


In [34]:
# Create mappings from source IDs to surrogate keys
patient_key_map = dim_patient.set_index("patient_id")["patient_key"]
doctor_key_map = dim_doctor.set_index("doctor_id")["doctor_key"]
clinic_key_map = dim_clinic.set_index("clinic_id")["clinic_key"]
date_key_map = dim_date.set_index("full_date")["date_key"]

# Start the fact table from cleaned appointments
fact_appointment = appointments_clean[
    [
        "appointment_id",
        "patient_id",
        "doctor_id",
        "clinic_id",
        "appointment_date",
        "wait_minutes",
        "consultation_minutes",
        "fee"
    ]
].copy()

# Replace source IDs with surrogate keys
fact_appointment["patient_key"] = (
    fact_appointment["patient_id"].map(patient_key_map)
)

fact_appointment["doctor_key"] = (
    fact_appointment["doctor_id"].map(doctor_key_map)
)

fact_appointment["clinic_key"] = (
    fact_appointment["clinic_id"].map(clinic_key_map)
)

fact_appointment["date_key"] = (
    fact_appointment["appointment_date"].map(date_key_map)
)

# Keep only the required fact-table columns
fact_appointment = fact_appointment[
    [
        "appointment_id",
        "patient_key",
        "doctor_key",
        "clinic_key",
        "date_key",
        "wait_minutes",
        "consultation_minutes",
        "fee"
    ]
]

print("fact_appointment rows:", len(fact_appointment))
display(fact_appointment.head())

fact_appointment rows: 1500


,appointment_id,patient_key,doctor_key,clinic_key,date_key,wait_minutes,consultation_minutes,fee
0,A00465,201,15,5,20250425,3,28,3500.0
1,A00270,11,19,1,20250310,2,16,2500.0
2,A00976,97,20,5,20250819,13,29,4500.0
3,A00657,255,4,4,20250609,2,10,3000.0
4,A00983,70,15,5,20250820,40,30,3500.0


In [35]:
print("Missing surrogate keys:")
print(
    fact_appointment[
        ["patient_key", "doctor_key", "clinic_key", "date_key"]
    ].isna().sum()
)

Missing surrogate keys:
patient_key    0
doctor_key     0
clinic_key     0
date_key       0
dtype: int64


In [36]:
import os

# Create output folder
os.makedirs("warehouse_data", exist_ok=True)

# Export warehouse tables
dim_patient.to_csv(
    "warehouse_data/dim_patient.csv",
    index=False
)

dim_doctor.to_csv(
    "warehouse_data/dim_doctor.csv",
    index=False
)

dim_clinic.to_csv(
    "warehouse_data/dim_clinic.csv",
    index=False
)

dim_date.to_csv(
    "warehouse_data/dim_date.csv",
    index=False
)

fact_appointment.to_csv(
    "warehouse_data/fact_appointment.csv",
    index=False
)

print("Warehouse CSV files exported successfully.")

Warehouse CSV files exported successfully.


In [37]:
print("dim_patient:", len(dim_patient))
print("dim_doctor:", len(dim_doctor))
print("dim_clinic:", len(dim_clinic))
print("dim_date:", len(dim_date))
print("fact_appointment:", len(fact_appointment))

dim_patient: 300
dim_doctor: 20
dim_clinic: 5
dim_date: 365
fact_appointment: 1500


In [38]:
patient_updates

,patient_id,new_district,effective_date
0,P0022,Matara,2026-01-15
1,P0103,Colombo,2026-01-15
2,P0233,Kalutara,2026-01-15


### Task 7.1 – Identify Patient District Changes

The `patient_updates` table contains changes to patient districts that become effective on 15 January 2026. Since the coursework analyses appointment data from 2025, the original district is the district that was valid during the 2025 appointment period.

First, the original district is joined to the update records so that the old and new district values can be compared.

In [39]:
patient_updates_with_old = patient_updates.merge(
    patients[["patient_id", "district"]],
    on="patient_id",
    how="left"
)

patient_updates_with_old

,patient_id,new_district,effective_date,district
0,P0022,Matara,2026-01-15,Jaffna
1,P0103,Colombo,2026-01-15,Galle
2,P0233,Kalutara,2026-01-15,Jaffna


### Task 7.2 – Overwrite Approach

In the overwrite approach, the patient's current district is replaced with the new district from `patient_updates`. This means the updated district is used even when analysing historical 2025 appointments.

The purpose is to show how the 2025 appointment counts by district would change if the historical district values were overwritten.

In [41]:
# Create a copy so the original patient data is not changed
patients_overwrite = patients.copy()

# Replace the district with the new district for updated patients
patients_overwrite = patients_overwrite.merge(
    patient_updates[["patient_id", "new_district"]],
    on="patient_id",
    how="left"
)

patients_overwrite["district"] = patients_overwrite["new_district"].fillna(
    patients_overwrite["district"]
)

patients_overwrite = patients_overwrite.drop(columns=["new_district"])

patients_overwrite.head()

,patient_id,gender,date_of_birth,district
0,P0001,F,1980-06-21,Kurunegala
1,P0002,F,2006-12-04,Gampaha
2,P0003,M,1959-02-14,Galle
3,P0004,M,1990-02-09,Matara
4,P0005,F,1975-01-20,Colombo


### 2025 Appointment Counts Before and After the Overwrite

The original patient district values are used to calculate the number of 2025 appointments by district. The same appointments are then joined with the overwritten patient dimension to calculate the counts using the new district values.

Comparing these two results shows which districts gain or lose historical appointment counts when patient districts are overwritten.

In [42]:
# Select the 2025 appointment records
appointments_2025 = appointments_clean[
    appointments_clean["appointment_date"].dt.year == 2025
].copy()

# Count 2025 appointments using the original districts
before = (
    appointments_2025
    .merge(
        patients[["patient_id", "district"]],
        on="patient_id",
        how="left"
    )
    .groupby("district")
    .size()
    .reset_index(name="appointments_before")
)

# Count 2025 appointments using overwritten districts
after = (
    appointments_2025
    .merge(
        patients_overwrite[["patient_id", "district"]],
        on="patient_id",
        how="left"
    )
    .groupby("district")
    .size()
    .reset_index(name="appointments_after")
)

# Compare both results
district_comparison = (
    before
    .merge(after, on="district", how="outer")
    .fillna(0)
)

district_comparison["change"] = (
    district_comparison["appointments_after"]
    - district_comparison["appointments_before"]
)

district_comparison.sort_values("district")

,district,appointments_before,appointments_after,change
0,Colombo,198,203,5
1,Galle,157,152,-5
2,Gampaha,242,242,0
3,Jaffna,210,199,-11
4,Kalutara,175,180,5
5,Kandy,191,191,0
6,Kurunegala,127,127,0
7,Matara,200,206,6


### Interpretation of the Overwrite Results

The overwrite approach changes the historical 2025 appointment counts because the three patients' original districts are replaced by their new districts.

Patient P0022 changes from Jaffna to Matara, patient P0103 changes from Galle to Colombo, and patient P0233 changes from Jaffna to Kalutara. Therefore, Jaffna decreases by 11 appointments, Galle decreases by 5 appointments, Colombo increases by 5 appointments, Kalutara increases by 5 appointments, and Matara increases by 6 appointments.

This demonstrates that overwriting patient district values can distort historical reporting because 2025 appointments are reassigned using district information that only became effective in 2026.

### Task 7.5 – Keep-History Approach

In the keep-history approach, the patient's district is not overwritten in the historical patient record. Instead, the district that was valid when the appointment occurred is retained for historical reporting.

For this dataset, all three district changes became effective on 15 January 2026. Therefore, the original districts should remain associated with the patients' 2025 appointments.

A dimensional model can support this approach using a slowly changing dimension (SCD) strategy, where a new dimension record is created when a patient's district changes. This preserves both the previous and new district values with their effective periods.

A clinic group may choose this approach because historical reports should remain consistent with the information that was valid at the time of each appointment. It allows management to compare historical performance accurately without later patient updates changing past results.

### Keep-History Verification

Because all three district updates are effective from 15 January 2026, they do not apply to appointments made during 2025.

Therefore, the 2025 appointment counts should remain based on the original patient districts. The result is compared with the overwrite approach to demonstrate the effect of preserving historical district information.

In [43]:
# Keep-history uses the original patient district values
keep_history = before.copy()

keep_history = keep_history.rename(
    columns={"appointments_before": "appointments_keep_history"}
)

keep_history = keep_history[
    ["district", "appointments_keep_history"]
]

keep_history.sort_values("district")

,district,appointments_keep_history
0,Colombo,198
1,Galle,157
2,Gampaha,242
3,Jaffna,210
4,Kalutara,175
5,Kandy,191
6,Kurunegala,127
7,Matara,200
